# 🎤 JOTACHI KARAOKE · Servidor de IA

Este cuaderno deja listo, **sin necesidad de PC**, el servidor que quita la voz de tus canciones y genera letras con tiempos.

**Cómo usarlo (desde el teléfono):**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU** (gratis, hace todo mucho más rápido).
2. Toca **Entorno de ejecución → Ejecutar todo**.
3. Al terminar verás una **URL** y una **clave**. Cópialas en la app: **Ajustes → URL del servidor / Clave API → Guardar y probar**.
4. **No cierres esta pestaña** mientras uses la app. Si Colab se desconecta, vuelve a ejecutar todo y copia la URL nueva (cambia cada vez).

In [ ]:
# 1) Comprobar GPU e instalar dependencias (tarda unos minutos la primera vez)
!nvidia-smi -L || echo "⚠️ Sin GPU: cambia el tipo de entorno a T4 GPU (si no, tardará mucho)."
!pip -q install torch==2.5.1 torchaudio==2.5.1 soundfile
!pip -q install demucs fastapi "uvicorn[standard]" python-multipart faster-whisper

In [ ]:
# 2) Descargar el código del servidor desde tu repositorio de GitHub
import os, pathlib, sys, time, re, secrets, subprocess, urllib.request, zipfile

SRC_URL = "https://raw.githubusercontent.com/jhonatannh19-tech/JOTACHI-KARAOKE/main/jotachi_karaoke.zip"
urllib.request.urlretrieve(SRC_URL, "src.zip")
zipfile.ZipFile("src.zip").extractall("src")
server_dir = next(pathlib.Path("src").glob("**/server/main.py")).parent.resolve()
assert (server_dir / "lyrics_utils.py").exists(), "El zip del repositorio es antiguo: sube la versión nueva de jotachi_karaoke.zip"
print("Servidor en:", server_dir)

In [ ]:
# 3) Arrancar el servidor
API_KEY = secrets.token_urlsafe(12)
env = dict(os.environ, API_KEY=API_KEY, JOBS_DIR="/content/jobs")
srv_log = open("/content/server.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=str(server_dir), env=env, stdout=srv_log, stderr=subprocess.STDOUT)

ok = False
for _ in range(40):
    time.sleep(1)
    try:
        req = urllib.request.Request("http://127.0.0.1:8000/health", headers={"X-API-Key": API_KEY})
        if urllib.request.urlopen(req, timeout=2).status == 200:
            ok = True
            break
    except Exception:
        pass
if not ok:
    print(open("/content/server.log").read()[-2000:])
    raise RuntimeError("El servidor no arrancó (mira el registro de arriba)")
print("Servidor local listo ✔")

In [ ]:
# 4) Abrir un túnel público gratuito (Cloudflare) para que el teléfono lo alcance
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "/content/cloudflared")
os.chmod("/content/cloudflared", 0o755)
cf_log = open("/content/cf.log", "w")
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=cf_log, stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(90):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cf.log").read())
    if m:
        PUBLIC_URL = m.group(0)
        break
if not PUBLIC_URL:
    print(open("/content/cf.log").read()[-2000:])
    raise RuntimeError("No se pudo abrir el túnel")

print("=" * 50)
print("COPIA ESTO EN LA APP (Ajustes):")
print("URL del servidor :", PUBLIC_URL)
print("Clave API        :", API_KEY)
print("=" * 50)
print("Deja esta celda en ejecución y NO cierres la pestaña.")

In [ ]:
# 5) Mantener el servidor vivo (esta celda no termina; detenla con ■ para apagar)
try:
    while True:
        time.sleep(30)
        if server.poll() is not None:
            print("El servidor se detuvo. Registro:")
            print(open("/content/server.log").read()[-2000:])
            break
except KeyboardInterrupt:
    pass
finally:
    server.terminate()
    tunnel.terminate()